# Notes

<u>System questions/notes:</u>
* What is a cluster? Do we need replicas?
* Collection - do we need partitions?
* The schema seems automatically determined based on the attributes and fields of the Documents. AutoID and dynamic field are N/A (`False` and `False`) when using Milvus with Haystack.
    ```python
    assert doc.id == res[0]["id"] # primary field
    assert doc.embedding == res[0]["vector"] # embedding
    assert doc.content == res[0]["text"] # text
    assert doc.meta["source_id"] == res[0]["source_id"] # autogenerated file ID
    assert doc.meta["page_number"] == res[0]["page_number"] # file page number
    assert doc.meta["split_id"] == res[0]["split_id"] # split index
    assert doc.meta["split_idx_start"] == res[0]["split_idx_start"] # not sure what this means
    assert doc.meta["file_path"] == res[0]["file_path"] # file
    ```
    * Primary field needs to be unique, but if I'm adding documents to an already existing collection, how do I know "id" and "source_id" will be unique?
    * doc.meta["_split_overlap"] was discarded with the error `has metadata fields with unsupported types: ['_split_overlap']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.`. I think this is because doc.meta["_split_overlap"] is `list[dict]`. This might be important for retrieval.
    * Is it possible for different file types to have different metadata? If so, how can we handle that?
* Figure out how AUTOINDEX works; understand all the different indexes and metrics
    * What kind of metric should we use? This depends on whether the SentenceTransformer embeddings are normalized. What's the best practice?
    * Can I change the metric after the collection is created? I'm guessing that yes, I can, but the indexing will have to be rerun since the indexing depends on the metric.
    * You should also index by tag. Zilliz uses **TRIE** for integers and **STL_SORT** for strings. Can I add an index after the collection is already created?
* Vector fields
    * Should we have any kind of metadata embedding? https://haystack.deepset.ai/tutorials/39_embedding_metadata_for_improved_retrieval
        * You can specify which metadata to embed when you initialize SentenceTransformer component - but it's based on keys, so if you want to do this, you may want metadata_cleaner after embedder
    * Multimodal embedding for images?
    * Utilize both dense and sparse embeddings, and search both using `hybrid_search()`?
* Pipelines
    * Indexing - this should be relatively straightforward
    * Tagging - get the new IDs that were indexed into the vector DB; pass them through the LLM to generate tags; add tag metadata to the new entities

<u>General questions:</u>
* If we know the course the learning material is from, why can't we get the tag directly based on the course/learning track? Guess: Courses may cover many subdisciplines; courses are not necessarily rigidly only one discipline.

<u>Zilliz notes:</u>
* Quickstart notes
    * The insert operations are asynchronous, and conducting a search immediately after data insertions may result in empty result set. To avoid this, you are advised to wait for a few seconds.
    * Searches are semantic searches (client.search), but you can also apply scalar field filters. Queries (client.query) are based on scalar filters only. You can also directly retrieve entities by their ID using client.get (instead of using query).
* Collection notes
    * You need to load a collection into memory to search and query it. This means loading the index files and the raw data of the fields.
    * A collection cannot be loaded without an index file
    * To reduce memory usage and improve search performance, you can specify which fields you want to load (instead of all fields)
        * Only these fields may be used for filtering and as outputs in search and query
        * You should always include the primary field and at least one vector field
    * Entities inserted after a collection load are automatically indexed and loaded
* Schema notes
    * You need a primary key and a vector field for every entity
    * To insert JSON data under `"metadata"` key, I think you need `enable_dynamic_field=True` when creating the schema
* Indexing notes
    * Recommended to create indexes for both vector field(s) and scalar field(s) that are frequently accessed.
    * Vector field indexes are for semantic search; scalar field indexes are for metadata filtering.
    * You can create up to one index per field in a collection.
    * You can always modify indexes by dropping the old index and adding a new one
    * For vector field indexes, Zilliz Cloud supports AUTOINDEX (https://docs.zilliz.com/docs/autoindex-explained)
        * Performance-optimized and capacity-optimized clusters require different approaches to indexing - AUTOINDEX takes care of that
        * Improved performance via SIMD, data graphing and cropping, and dynamic quantization
        * AUTOINDEX automatically chooses search parameters to trade off between recall and performance. Search params only has 1 parameter: level. Higher level = higher recall, but possibly slower search. Level defaults to 1 and ranges from 1 to 10. Default value = 90% recall. `enable_recall_calculation`?

<u>Action items:</u>
<s>
* Add tag as a field to the schema - `DataType.ARRAY` makes sense for this
    * If you define a schema field that isn't automatically generated by the Haystack pipeline, then a KeyError exception is thrown when you run the pipeline - in MilvusDocumentStore.write_documents, there is a line of code that iterates through the fields `insert_list = [insert_dict[x][i:end] for x in self.fields]`. `self.fields` is initialized via `_extract_fields()` that extracts fields from the collection schema, which includes `tags`, but since the pipeline doesn't generate a `tags` field, there is a KeyError.
    * I think you need to include `tags` when creating the collection. You can create it after, when you actually do the tagging, but you'll just run into the same exception since new documents will pass through the indexing pipeline.
    * Setting to nullable doesn't help since it's still part of the schema
    * Create a custom component to add this metadata to Document
    * There seems to be some issue with MilvusDocumentStore calling `pymilvus.orm.types.infer_dtype_bydata`. For `list[str]`, this returns `DataType.UNKNOWN` but should return `DataType.ARRAY`.
        * I am able to insert `list[str]` as an ARRAY using pymilvus, but this doesn't work with Haystack. `infer_dtype_bydata` returns `UNKNOWN` for `list[str]` and `FLOAT_VECTOR` for `list[int]`, neither of which are correct. For `list[int]`, creating the collection fails because it thinks that `"tags"` is a vector, and there's no `dim` param:
        ```python
        {'name': 'tags', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>}
        {'name': 'tags', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>}
        {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}
        {'name': 'id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}, 'is_primary': True, 'auto_id': False}
        {'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 3}}
        2025-02-25 08:44:46,884 [ERROR][handler]: RPC error: [create_collection], <MilvusException: (code=65535, message=dimension is not defined in field type params, check type param `dim` for vector field)>, <Time:{'RPC start': '2025-02-25 08:44:46.799552', 'RPC error': '2025-02-25 08:44:46.883781'}> (decorators.py:140)
        Failed to create collection: TestARRAYInsert error: <MilvusException: (code=65535, message=dimension is not defined in field type params, check type param `dim` for vector field)>
        ```
    * Try using a JSON object. Filtering with JSON fields
        * https://docs.zilliz.com/docs/use-json-fields
        * https://docs.zilliz.com/docs/json-filtering-operators
    * Create a custom component to wrap around pymilvus
* Check if `split_overlap` doc IDs match document IDs in the vector store (yes). Can you convert this to a JSON object or an ARRAY? For JSON, perhaps you can convert from list of dicts to just one dict.
* Can I put everything under Document.meta["metadata"]?
* How does Haystack determine document and source IDs? Are these guaranteed to be unique? I think it's fine to use the Haystack ID creation. It's under Document._create_id.
    * If no ID is provided when creating the document, then the ID is created using SHA-256, which produces a 256-bit (64-character) hash.
    * Converter creates a document ID, but then cleaner rehashes the document ID.
    * Splitter creates document IDs for each split, and the document ID from cleaner becomes metadata, stored under source_id. Splitter also stores the source_ids for overlapping chunks (metadata).
* Upserting (for tagging) and duplication policy - I think you need to rely on Haystack Document hash ID.
    * Milvus doesn't seem to support checking for duplicate primary keys, based on `MilvusDocumentStore.write_documents` and `MilvusClient.insert`. MilvusDocumentStore also doesn't seem to support auto ID - it explicitly checks for the document ID.
    * In this case, you may want to consider enabling auto ID for the primary key, if possible. It does mean that you can't use split_overlap_ids anymore, but you still have access to document IDs if you want.
* Try with synchronous tagging pipeline first (proof-of-concept). Start with parquet file (video transcripts)
    * Custom component for querying - Milvus retriever component only supports search (requires embedding)
        * If pass list of IDs, then uses MilvusClient.get()
        * Otherwise, uses MilvusClient.query() looking for empty tags
    * Custom component for looping through documents, building prompts, and generation
        * Init a pipeline: prompt builder -> generator
        * Run loops over the documents, calling pipeline on each. Take the pipeline output and upsert into the database.
    * Custom component for upserting Documents into the vector database
        * Use json.loads() to convert the generator output string to a list
        * This inherits from DocumentWriter - call DocumentWriter.run() instead of duplicating code
</s>
* Integrate RecursiveCharacterTextSplitter from LangChain into indexing pipeline
* Are SentenceTransformer embeddings normalized? This will affect which metric is appropriate for indexing and search
    * Can index metric and search metric be different? The Zilliz documentation says you can specify the search metric, so it seems independent of index metric...
* Index the transcripts
* Schema: IDs are 64 characters long
* Add component to check for ID collision? And then choose to throw exception or warning; overwrite, etc. I think most likely it makes sense to log a warning and move on (do not write anything).
    * MilvusDocumentStore.write_documents() uses MilvusClient.insert() only
* Do I need an index for tags? If so, then the current implementation, where it's stored under "metadata", won't work because dynamic fields do not support indexing. I don't think so - I think you only need indexing if you're going to use >, < operators.
    * https://docs.zilliz.com/docs/use-json-fields
    * https://docs.zilliz.com/docs/json-filtering-operators
* Any other metadata needed?
    * **Course / course number / course name**
    * Lecturer
    * Year / semester - track old content
    * Lecture number or lecture title
* Indexing pipeline should return all IDs that were just inserted to pass to tagging pipeline
* Look into recursive text splitter from LangChain
* How do we process different kinds of documents and ensure that they all make it through the pipeline? (e.g. could they have different metadata?)
* Do I need to make sure to close the Milvus connection when finished reading/writing?
* Should I use dependency injection for `dict_to_doc()`? This will probably be used everywhere.
* Check haystack/validators. If tagging output is not in the correct format, or if metadata isn't in the correct format. Or just try/except with looping (agent?)
* How can I enforce that Documents have the same structure in every part of the code? Can I do this with Pydantic?
* Start working on prompt engineering once you have the transcripts. Semi-supervised learning? Need some way to evaluate tagging accuracy (perhaps manual labeling of a subset).
* Start thinking about containerizing the pipeline(s) - one container per pipeline; one Lambda function per container?
* Pytest
* AsyncOpenAI, AsyncMilvusClient
    * Custom component that wraps around AsyncOpenAI client - pass all the chat completion requests and finish them asynchronously for speed
            * Check async pipeline template under api-example branch. utils/main.py; pipelines/agent_pipeline.py
            * https://haystack.deepset.ai/cookbook/async_pipeline
* "As of July 2024, gpt-4o-mini should be used in place of gpt-3.5-turbo, as it is cheaper, more capable, multimodal, and just as fast. gpt-3.5-turbo is still available for use in the API."

## api-example Directory Structure
```python
.
├── Dockerfile
├── Dockerfile.dev
├── README.md
├── app
│   ├── __init__.py
│   ├── data
│   │   └── data_models.py
│   ├── haystack_utilities
│   │   ├── __init__.py
│   │   ├── ml
│   │   │   ├── models.py
│   │   │   ├── prompt_meta.py
│   │   │   ├── prompt_meta_pdf.py
│   │   │   └── prompt_meta_sql.py
│   │   ├── pipelines
│   │   │   ├── agent_pipeline.py
│   │   │   ├── pdf_pipeline.py
│   │   │   └── sql_pipeline.py
│   │   └── tools
│   │       ├── async_tools.py
│   │       ├── sql_connector.py
│   │       └── tool_definitions.py
│   ├── main.py
│   └── utils
│       ├── auth.py
│       ├── aws_utils.py
│       ├── conversion_tools.py
│       └── observation.py
├── boot_scripts
│   └── boot.sh
├── examples
│   ├── milvus_haystack
│   │   ├── milvus_rag_standalone.py
│   │   ├── milvus_rag_standalone_conv.py
│   │   ├── models.py
│   │   ├── prompt_meta.py
│   │   ├── prompt_meta_conv.py
│   │   └── requirements.txt
│   └── milvus_qa
│       ├── README.md
│       ├── data
│       │   └── tokenizer.json
│       ├── images
│       │   └── bench_shot.png
│       ├── milvus_utils.py
│       ├── onnx_model.py
│       ├── question_answer.csv
│       └── requirements.txt
├── project_utilities
│   └── models_dl.py
├── requirements.txt
└── tests
    ├── __init__.py
    ├── e2e
    │   ├── __init__.py
    │   └── test_endpoints_auth.py
    ├── integration
    │   ├── __init__.py
    │   ├── test_async_pipe.py
    │   ├── test_conv_cache.py
    │   └── test_pdf_pipe.py
    ├── unit
    │   ├── __init__.py
    │   └── test_ml.py
    └── user_setup.py
```

# Zilliz/Milvus API

In [113]:
from pymilvus import MilvusClient, DataType
from dotenv import load_dotenv
import os

load_dotenv()

True

In [115]:
client = MilvusClient(
    uri=os.getenv("ZILLIZ_CLUSTER_ENDPOINT"),
    token=os.getenv("ZILLIZ_CLUSTER_TOKEN")
)

# client = MilvusClient(
#     uri="milvus.db"
# )

In [116]:
for col in client.list_collections():
    client.drop_collection(col)
print(client.list_collections())
client.close()

[]


## Collections

In [ ]:
# View collections

coll_name = client.list_collections()[0]
# print(coll_name)
# print("-"*20)
coll_description = client.describe_collection(coll_name)
for key, value in coll_description.items():
    if type(value) == list:
        print(f"{key}:")
        for v in value:
            print(v)
    else:
        print(f"{key}: {value}")
    print("-"*20)

In [ ]:
# Load collection into memory for search and query

client.load_collection(coll_name)
client.get_load_state(coll_name)

In [ ]:
# Release collection from memory

client.release_collection(coll_name)
client.get_load_state(coll_name)

## Indexes

In [ ]:
# List and describe indexes

index_name = client.list_indexes(coll_name)[0]
client.describe_index(coll_name, index_name)

In [ ]:
# Drop index

client.release_collection(coll_name)
client.get_load_state(coll_name)
client.drop_index(coll_name, index_name)
client.list_indexes(coll_name)

In [ ]:
# Add index

index_params = client.prepare_index_params()

index_params.add_index(
    field_name = "vector",
    metric_type = "L2",
    index_type = "AUTOINDEX",
    index_name = "vector_index"
)

client.create_index(coll_name, index_params)

# Haystack-Zilliz Indexing Pipeline
* Pipeline: converter -> cleaner -> splitter -> embedder -> metadata_cleaner (custom) -> writer
    * metadata_cleaner moves all metadata from `Document.meta` to `Document.meta["metadata"]` so writer can insert the metadata as a JSON object (otherwise ran into errors when trying to insert certain types of metadata)
    * metadata_cleaner also adds "tags" metadata - since there is no tag yet, it sets `Document.meta["metadata"]["tags"] = []`
* Primary key generation
    * cleaner generates `Document.id` - this identifies the file. At the end of the pipeline, this is stored under `Document.meta["metadata"]["source_id"]` because splitter generates new `Document.id` for each split.
    * The `Document.id` generated by splitter eventually becomes the primary key
    * It does not appear possible to enable auto ID when using Haystack-Milvus component, `MilvusDocumentStore.write_documents`
    * MilvusDocumentStore also does not support checking for duplicate primary key, so if there are multiple identical primary keys, they will all be inserted as unique entities.
    * Haystack ID generation is based on SHA-256 (`Document._create_id()`)
* Metadata
    * 'tags': TBD
    * 'source_id': ID corresponding to the output of cleaner (essentially an ID that identifies the original file)
    * 'file_path': file path/name
    * 'page_number'
    * 'split_overlap_ids': when `split_overlap` is enabled in the splitter, this identifies the overlapping (adjacent) chunks

In [1]:
from haystack import Pipeline, Document, component
from milvus_haystack import MilvusDocumentStore
from haystack.components.converters import PyPDFToDocument
from haystack.components.preprocessors import DocumentCleaner, DocumentSplitter
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.writers import DocumentWriter
from haystack.utils import Secret
from pymilvus import MilvusClient, DataType
from typing import List

In [2]:
client = MilvusClient(
    uri = Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
    token = Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),
)

In [10]:
# https://docs.zilliz.com/docs/manage-collections-sdks
# https://milvus.io/docs/array_data_type.md
# https://milvus.io/docs/dense-vector.md

schema = MilvusClient.create_schema(
    auto_id=False,
    enable_dynamic_field=True,
)

schema.add_field(field_name="id", datatype=DataType.VARCHAR, is_primary=True, auto_id=False, max_length=512)
schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=768)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=65535)
schema.add_field(field_name="metadata", datatype=DataType.JSON)
# schema.add_field(field_name="tags", datatype=DataType.ARRAY, element_type=DataType.VARCHAR, max_capacity=10, max_length=20, nullable=True)
# schema.add_field(field_name="split_overlap_ids", datatype=DataType.ARRAY, element_type=DataType.VARCHAR, max_capacity=2, max_length=512, nullable=True)

index_params = client.prepare_index_params()

index_params.add_index(
    field_name = "vector",
    metric_type = "COSINE",
    index_type = "AUTOINDEX",
    index_name = "vector_index"
)

if "HaystackCollection" in client.list_collections():
    client.drop_collection("HaystackCollection")
    
client.create_collection(
    collection_name = "HaystackCollection",
    schema = schema,
    index_params = index_params,
)

client.get_load_state("HaystackCollection")

{'state': <LoadState: Loaded>}

In [11]:
# Connect to Milvus client and create new collection

# file_names = ["Project Management Requirements Handbook.pdf"]

# document_store = MilvusDocumentStore(
#     collection_name = "HaystackCollection",
#     collection_description = "Test collection",
#     collection_properties = None,
#     connection_args = {
#         # "uri": "https://in01-65d96e8e6d6e34c.aws-us-east-1.vectordb.zillizcloud.com:19539",  # Public Endpoint
#         "uri": Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
#         "token": Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),  # API key.
#         "secure": True
#         },
#     consistency_level = "Strong", # Strong, Bounded, Eventually, Session
#     index_params = {
#         "index_type": "AUTOINDEX",
#         "metric_type": "COSINE",
#     },
#     search_params = {
#         "params": {
#             "level": 1
#         }
#     },
#     drop_old = True,
# )

# Connect to Milvus client and add to an existing collection

file_names = ["Project Management Requirements Handbook.pdf", "Lec1 Machine Learning Review.pdf"]

document_store = MilvusDocumentStore(
    collection_name = "HaystackCollection",
    # collection_description = "Test collection",
    # collection_properties = None,
    connection_args = {
        # "uri": "https://in01-65d96e8e6d6e34c.aws-us-east-1.vectordb.zillizcloud.com:19539",  # Public Endpoint
        "uri": Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
        "token": Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),  # API key.
        "secure": True
        },
    # consistency_level = "Strong", # Strong, Bounded, Eventually, Session
    # index_params = {
    #     "index_type": "AUTOINDEX",
    #     "metric_type": "COSINE",
    # },
    # search_params = {
    #     "params": {
    #         "level": 1
    #     }
    # },
    # drop_old=False,
)

In [13]:
# Create custom component to handle _split_overlap and to add null tags

add_MetadataCleaner = True

@component
class MetadataCleaner:
    @component.output_types(documents=List[Document])
    def run(self, documents: List[Document]):
        docs = [self._add_metadata(doc) for doc in documents]
        return {"documents": documents}
    
    def _add_metadata(self, doc: Document) -> Document:
        # Initialize the new metadata dictionary
        doc.meta["metadata"] = {
            "tags": [],
            "source_id": doc.meta.pop("source_id", None),
            "file_path": doc.meta.pop("file_path", None),
            "page_number": doc.meta.pop("page_number", None),
            "split_overlap_ids": [d["doc_id"] for d in doc.meta.pop("_split_overlap", [])]
        }

        # Remove all keys except "metadata"
        keys_to_remove = [key for key in list(doc.meta) if key != "metadata"]
        for key in keys_to_remove:
            doc.meta.pop(key)

        return doc

In [14]:
# Create indexing pipeline

pipe = Pipeline()

pipe.add_component("converter", PyPDFToDocument(extraction_mode="layout"))
pipe.add_component("cleaner", DocumentCleaner())
pipe.add_component("splitter", DocumentSplitter(split_by="word", split_length=100, split_overlap=10, split_threshold=50))
if add_MetadataCleaner:
    pipe.add_component("metadata_cleaner", MetadataCleaner())
pipe.add_component("embedder", SentenceTransformersDocumentEmbedder())
pipe.add_component("writer", DocumentWriter(document_store=document_store))

pipe.connect("converter", "cleaner")
pipe.connect("cleaner", "splitter")
# if add_MetadataCleaner:
#     pipe.connect("splitter", "metadata_cleaner")
#     pipe.connect("metadata_cleaner", "embedder")
# else:
#     pipe.connect("splitter", "embedder")
pipe.connect("splitter", "embedder")
if add_MetadataCleaner:
    pipe.connect("embedder", "metadata_cleaner")
    pipe.connect("metadata_cleaner", "writer")
else:
    pipe.connect("embedder", "writer")

In [15]:
if add_MetadataCleaner:
    last_component = "metadata_cleaner"
else:
    last_component = "embedder"

results = pipe.run({"converter": {"sources": file_names}}, include_outputs_from={last_component})

Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text discovered. Output will be incomplete.
Rotated text

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [16]:
print(document_store.count_documents())
print(document_store.fields)

70
['id', 'vector', 'text', 'metadata']


In [18]:
# Sanity-check embedder output

idx = 1
doc = results[last_component]["documents"][idx]
print(doc)
print(f"id: {doc.id}")
for k, v in doc.meta.items():
    print(f"{k}: {v}")

Document(id=4b2a525a6a7bb7ed28badb0ff22fdb31717c2ef1ae95f90d8b3ccd3115ec31db, content: '2.3 Election ofthe team leader 10 3.PROJECT COMMUNICATIONS 10 3.1 Onboarding Training 10 3.2 Emails,...', meta: {'metadata': {'tags': [], 'source_id': '8ac81db781a7877b09869dd6be3c0fdd7a4a737864c9f647bb3cfe3af66cb71e', 'file_path': 'Project Management Requirements Handbook.pdf', 'page_number': 2, 'split_overlap_ids': ['897ba1906797de32cfea2c683fb590d1733ea0be2bf7bd7d00541923937d6f9a', 'af6d2261b0b0b92ec74d222bb3665b639be914f0ca45708aadd4af6572c09aa0']}}, embedding: vector of size 768)
id: 4b2a525a6a7bb7ed28badb0ff22fdb31717c2ef1ae95f90d8b3ccd3115ec31db
metadata: {'tags': [], 'source_id': '8ac81db781a7877b09869dd6be3c0fdd7a4a737864c9f647bb3cfe3af66cb71e', 'file_path': 'Project Management Requirements Handbook.pdf', 'page_number': 2, 'split_overlap_ids': ['897ba1906797de32cfea2c683fb590d1733ea0be2bf7bd7d00541923937d6f9a', 'af6d2261b0b0b92ec74d222bb3665b639be914f0ca45708aadd4af6572c09aa0']}


In [205]:
# Describe the collection (schema)

col_name = document_store.collection_name

print(document_store.client.get_load_state(col_name))
col_description = document_store.client.describe_collection(col_name)
for k, v in col_description.items():
    if type(v) == list:
        for d in v:
            print(d)
    else:
        print(f"{k}: {v}")

{'state': <LoadState: Loaded>}
collection_name: HaystackCollection
auto_id: False
num_shards: 1
description: 
{'field_id': 100, 'name': 'id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 512}, 'is_primary': True}
{'field_id': 101, 'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}
{'field_id': 102, 'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}
{'field_id': 103, 'name': 'metadata', 'description': '', 'type': <DataType.JSON: 23>, 'params': {}}
collection_id: 456094611421093023
consistency_level: 2
properties: {}
num_partitions: 1
enable_dynamic_field: True


In [156]:
# Check how the Documents are written into the database (using MilvusClient.query)

expr = "id == {id}"
counter = 0
for doc in results["embedder"]["documents"]:
    filter_params = {"id": doc.id}
    res = document_store.client.query(
        collection_name = col_name,
        filter = expr,
        output_fields = ["*"],
        filter_params = filter_params
    )
    assert len(res) == 1, "Document IDs should map 1:1 to id in the collection"
    assert doc.id == res[0]["id"]
    assert doc.embedding == res[0]["vector"]
    assert doc.content == res[0]["text"]
    # assert doc.meta["source_id"] == res[0]["source_id"]
    # assert doc.meta["page_number"] == res[0]["page_number"]
    # assert doc.meta["split_id"] == res[0]["split_id"]
    # assert doc.meta["split_idx_start"] == res[0]["split_idx_start"]
    # assert doc.meta["file_path"] == res[0]["file_path"]

    for k, v in doc.meta.items():
        if k in res[0]:
            assert doc.meta[k] == res[0][k]

    counter += 1
    print(counter)
    # break

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70


In [ ]:
# Check if doc.meta is the same for all documents (using DocumentStore.filter_documents)
doc = results["embedder"]["documents"][0]

# for field in ["source_id", "page_number", "split_id", "split_idx_start", "file_path"]:
#     expr = f"{field} == " + "{value}"
#     filter_params = {"value": doc.meta[field]}
#     res = document_store.client.query(
#         collection_name = col_name,
#         filter = expr,
#         output_fields = ["*"],
#         filter_params = filter_params
#     )
#     print(len(res))

for field in ["source_id", "page_number", "split_id", "split_idx_start", "file_path"]:
    filters = {"field": f"meta.{field}", "operator": "==", "value": doc.meta[field]}
    # print(filters)
    res = document_store.filter_documents(filters)
    print(len(res))

In [ ]:
# Check doc.meta for all documents

for idx, doc in enumerate(results["embedder"]["documents"]):
    print(idx)
    for k, v in doc.meta.items():
        print(f"{k}: {v}")
    print("-"*30)

In [ ]:
filters = {"field": "meta.source_id", "operator": "==", "value": "cf691c062fc851a2ed51fa786741e86677f0f44e6004c4bff49e5c2dec264f67"}
filters = {"field": "meta.source_id", "operator": "==", "value": "8ac81db781a7877b09869dd6be3c0fdd7a4a737864c9f647bb3cfe3af66cb71e"}
res = document_store.filter_documents(filters)
print(len(res))
# res

# Inserting ARRAY Into Collection
## Using pymilvus

In [ ]:
# https://docs.zilliz.com/docs/use-array-fields

client = MilvusClient(
    uri = Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
    token = Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),
)

In [ ]:
schema = client.create_schema(
    auto_id = True,
    enable_dynamic_field = True,
)

schema.add_field(
    field_name = "pk",
    datatype = DataType.VARCHAR,
    max_length = 512,
    is_primary = True
)
schema.add_field(
    field_name = "tags",
    datatype = DataType.ARRAY,
    element_type = DataType.VARCHAR,
    max_capacity = 10,
    max_length = 65535
)
schema.add_field(
    field_name = "vector",
    datatype = DataType.FLOAT_VECTOR,
    dim = 3
)

index_params = client.prepare_index_params()
index_params.add_index(
    field_name = "tags",
    index_type = "AUTOINDEX"
)
index_params.add_index(
    field_name="vector",
    index_type="AUTOINDEX",
    metric_type="COSINE"
)

In [41]:
if "TestARRAYInsert" in client.list_collections():
    client.drop_collection("TestARRAYInsert")
    
client.create_collection(
    collection_name = "TestARRAYInsert",
    schema = schema,
    index_params = index_params
)

In [28]:
data = [
    {"tags": ["pop", "rock"],
     "vector": [0.1, 0.1, 0.1]},
    {"tags": ["pop", "rock"],
     "vector": [0.1, 0.1, 0.1]},
    {"tags": ["pop", "rock"],
     "vector": [0.1, 0.1, 0.1]},
]

In [23]:
client.insert(
    collection_name="TestARRAYInsert",
    data=data,
)

{'insert_count': 3, 'ids': ['456094611417375218', '456094611417375219', '456094611417375220'], 'cost': 0}

## Using Haystack

In [72]:
document_store = MilvusDocumentStore(
    collection_name = "TestARRAYInsert",
    connection_args = {
        "uri": Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
        "token": Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),
        "secure": True
        },
    drop_old=True,
)

writer = DocumentWriter(document_store=document_store)

In [74]:
# data = [
#     {"id": "1",
#      "meta": {"tags": CustomList(["pop", "rock"])},
#      "embedding": [0.1, 0.1, 0.1]},
#     {"id": "2",
#      "meta": {"tags": CustomList(["pop", "rock"])},
#      "embedding": [0.1, 0.1, 0.1]},
#     {"id": "3",
#      "meta": {"tags": CustomList(["pop", "rock"])},
#      "embedding": [0.1, 0.1, 0.1]},
# ]

# data = [
#     {"id": "1",
#      "meta": {"tags": ["pop", "rock"]},
#      "embedding": [0.1, 0.1, 0.1]},
#     {"id": "2",
#      "meta": {"tags": ["pop", "rock"]},
#      "embedding": [0.1, 0.1, 0.1]},
#     {"id": "3",
#      "meta": {"tags": ["pop", "rock"]},
#      "embedding": [0.1, 0.1, 0.1]},
# ]

data = [
    {"id": "1",
     "tags": ["pop", "rock"],
     "embedding": [0.1, 0.1, 0.1]},
    {"id": "2",
     "tags": ["pop", "rock"],
     "embedding": [0.1, 0.1, 0.1]},
    {"id": "3",
     "tags": ["pop", "rock"],
     "embedding": [0.1, 0.1, 0.1]},
]

# data = [
#     {"id": "1",
#      "meta": {"tags": [1, 2]},
#      "embedding": [0.1, 0.1, 0.1]},
#     {"id": "2",
#      "meta": {"tags": [1, 2]},
#      "embedding": [0.1, 0.1, 0.1]},
#     {"id": "3",
#      "meta": {"tags": [1, 2]},
#      "embedding": [0.1, 0.1, 0.1]},
# ]

docs = [Document.from_dict(d) for d in data]
docs

[Document(id=1, meta: {'tags': ['pop', 'rock']}, embedding: vector of size 3),
 Document(id=2, meta: {'tags': ['pop', 'rock']}, embedding: vector of size 3),
 Document(id=3, meta: {'tags': ['pop', 'rock']}, embedding: vector of size 3)]

In [76]:
writer.run(docs)

Document 1 has metadata fields with unsupported types: ['tags']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.
Document 2 has metadata fields with unsupported types: ['tags']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.
Document 3 has metadata fields with unsupported types: ['tags']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.


{'documents_written': 3}

In [53]:
from pandas.api.types import (
    infer_dtype,
    is_array_like,
    is_float,
    is_list_like,
    is_scalar,
)

In [63]:
myvar = ["a", "b"]
myvar = iter(["a", "b"])
is_list_like(data)

False

# Inserting JSON into Collection

## Using pymilvus

In [ ]:
# https://docs.zilliz.com/docs/use-json-fields

client = MilvusClient(
    uri = Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
    token = Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),
)

In [83]:
schema = client.create_schema(
    auto_id = True,
    enable_dynamic_field = True,
)

schema.add_field(
    field_name = "pk",
    datatype = DataType.VARCHAR,
    max_length = 512,
    is_primary = True
)
schema.add_field(field_name="metadata", datatype=DataType.JSON)
schema.add_field(
    field_name = "vector",
    datatype = DataType.FLOAT_VECTOR,
    dim = 3
)

index_params = client.prepare_index_params()

index_params.add_index(
    field_name="vector",
    index_type="AUTOINDEX",
    metric_type="COSINE"
)

In [88]:
if "TestJSONInsert" in client.list_collections():
    client.drop_collection("TestJSONInsert")
    
client.create_collection(
    collection_name = "TestJSONInsert",
    schema = schema,
    index_params = index_params
)

In [89]:
data = [
    {"metadata": {"tags": ["pop", "rock"]},
     "vector": [0.1, 0.1, 0.1]},
    {"metadata": {"tags": ["pop", "punk"]},
     "vector": [0.1, 0.1, 0.1]},
    {"metadata": {"tags": ["reggae", "hip hop"]},
     "vector": [0.1, 0.1, 0.1]},
]

In [90]:
client.insert(
    collection_name="TestJSONInsert",
    data=data,
)

{'insert_count': 3, 'ids': ['456094611417375250', '456094611417375251', '456094611417375252'], 'cost': 0}

In [104]:
res = client.search(
    collection_name="TestJSONInsert",
    data = [[0.1, 0.1, 0.1]],
    limit=5,
    output_fields=["*"],
    filter='json_contains(metadata["tags"], "pop")'
)

for r in res[0]:
    print(r)

{'pk': '456094611417375251', 'distance': 1.0, 'entity': {'pk': '456094611417375251', 'metadata': {'tags': ['pop', 'punk']}, 'vector': [0.10000000149011612, 0.10000000149011612, 0.10000000149011612]}}
{'pk': '456094611417375250', 'distance': 1.0, 'entity': {'pk': '456094611417375250', 'metadata': {'tags': ['pop', 'rock']}, 'vector': [0.10000000149011612, 0.10000000149011612, 0.10000000149011612]}}


## Using Haystack

In [106]:
document_store = MilvusDocumentStore(
    collection_name = "TestJSONInsert",
    connection_args = {
        "uri": Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
        "token": Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),
        "secure": True
        },
    drop_old=True,
)

writer = DocumentWriter(document_store=document_store)

In [109]:
data = [
    {"id": "1",
     "metadata": {"tags": ["pop", "rock"]},
     "embedding": [0.1, 0.1, 0.1]},
    {"id": "2",
     "metadata": {"tags": ["pop", "punk"]},
     "embedding": [0.1, 0.1, 0.1]},
    {"id": "3",
     "metadata": {"tags": ["reggae", "hip hop"]},
     "embedding": [0.1, 0.1, 0.1]},
]

docs = [Document.from_dict(d) for d in data]
docs

[Document(id=1, meta: {'metadata': {'tags': ['pop', 'rock']}}, embedding: vector of size 3),
 Document(id=2, meta: {'metadata': {'tags': ['pop', 'punk']}}, embedding: vector of size 3),
 Document(id=3, meta: {'metadata': {'tags': ['reggae', 'hip hop']}}, embedding: vector of size 3)]

In [110]:
writer.run(docs)

{'documents_written': 3}

In [111]:
document_store.client.describe_collection("TestJSONInsert")

{'collection_name': 'TestJSONInsert',
 'auto_id': False,
 'num_shards': 1,
 'description': '',
 'fields': [{'field_id': 100,
   'name': 'metadata',
   'description': '',
   'type': <DataType.JSON: 23>,
   'params': {}},
  {'field_id': 101,
   'name': 'text',
   'description': '',
   'type': <DataType.VARCHAR: 21>,
   'params': {'max_length': 65535}},
  {'field_id': 102,
   'name': 'id',
   'description': '',
   'type': <DataType.VARCHAR: 21>,
   'params': {'max_length': 65535},
   'is_primary': True},
  {'field_id': 103,
   'name': 'vector',
   'description': '',
   'type': <DataType.FLOAT_VECTOR: 101>,
   'params': {'dim': 3}}],
 'functions': [],
 'aliases': [],
 'collection_id': 456094611421047406,
 'consistency_level': 1,
 'properties': {},
 'num_partitions': 1,
 'enable_dynamic_field': False}

# Pipeline Component Outputs

In [181]:
file_names = ["Project Management Requirements Handbook.pdf"]

converter = PyPDFToDocument(extraction_mode="layout")
converter_output = converter.run(file_names)
print(converter_output["documents"][0])

Document(id=a23a6652ff5e96c4d547a6cc6d432f6d79bf1426f070fa567bb418f1fa93f774, content: '                                                 weclouddata.com




Client  Project    Management  ...', meta: {'file_path': 'Project Management Requirements Handbook.pdf'})


In [182]:
cleaner = DocumentCleaner()
cleaner_output = cleaner.run(converter_output["documents"])
print(cleaner_output["documents"][0])

Document(id=8ac81db781a7877b09869dd6be3c0fdd7a4a737864c9f647bb3cfe3af66cb71e, content: 'weclouddata.com Client Project Management Requirements Handbook [Student Version] BeamData Ltd, 180 ...', meta: {'file_path': 'Project Management Requirements Handbook.pdf'})


In [183]:
splitter = DocumentSplitter(split_by="word", split_length=100, split_overlap=10, split_threshold=50)
splitter_output = splitter.run(cleaner_output["documents"])
print(len(splitter_output["documents"]))
print(splitter_output["documents"][0])

40
Document(id=897ba1906797de32cfea2c683fb590d1733ea0be2bf7bd7d00541923937d6f9a, content: 'weclouddata.com Client Project Management Requirements Handbook [Student Version] BeamData Ltd, 180 ...', meta: {'file_path': 'Project Management Requirements Handbook.pdf', 'source_id': '8ac81db781a7877b09869dd6be3c0fdd7a4a737864c9f647bb3cfe3af66cb71e', 'page_number': 1, 'split_id': 0, 'split_idx_start': 0, '_split_overlap': [{'doc_id': '4b2a525a6a7bb7ed28badb0ff22fdb31717c2ef1ae95f90d8b3ccd3115ec31db', 'range': (0, 66)}]})


In [184]:
splitter_output["documents"][0].meta["source_id"] == cleaner_output["documents"][0].id

True

In [185]:
metadatacleaner = MetadataCleaner()
metadatacleaner_output = metadatacleaner.run(splitter_output["documents"])
print(len(metadatacleaner_output["documents"]))
print(metadatacleaner_output["documents"][0])

40
Document(id=897ba1906797de32cfea2c683fb590d1733ea0be2bf7bd7d00541923937d6f9a, content: 'weclouddata.com Client Project Management Requirements Handbook [Student Version] BeamData Ltd, 180 ...', meta: {'metadata': {'tags': None, 'source_id': '8ac81db781a7877b09869dd6be3c0fdd7a4a737864c9f647bb3cfe3af66cb71e', 'file_path': 'Project Management Requirements Handbook.pdf', 'page_number': 1, 'split_overlap_ids': ['4b2a525a6a7bb7ed28badb0ff22fdb31717c2ef1ae95f90d8b3ccd3115ec31db']}})


In [188]:
metadatacleaner_output["documents"][0].meta["metadata"]["source_id"] == cleaner_output["documents"][0].id

True

In [190]:
embedder = SentenceTransformersDocumentEmbedder()
embedder.warm_up()
embedder_output = embedder.run(metadatacleaner_output["documents"])
print(len(embedder_output["documents"]))
print(embedder_output["documents"][0])

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

40
Document(id=897ba1906797de32cfea2c683fb590d1733ea0be2bf7bd7d00541923937d6f9a, content: 'weclouddata.com Client Project Management Requirements Handbook [Student Version] BeamData Ltd, 180 ...', meta: {'metadata': {'tags': None, 'source_id': '8ac81db781a7877b09869dd6be3c0fdd7a4a737864c9f647bb3cfe3af66cb71e', 'file_path': 'Project Management Requirements Handbook.pdf', 'page_number': 1, 'split_overlap_ids': ['4b2a525a6a7bb7ed28badb0ff22fdb31717c2ef1ae95f90d8b3ccd3115ec31db']}}, embedding: vector of size 768)


In [ ]:
pipe.add_component("cleaner", DocumentCleaner())
pipe.add_component("splitter", DocumentSplitter(split_by="word", split_length=100, split_overlap=10, split_threshold=50))
if add_MetadataCleaner:
    pipe.add_component("metadata_cleaner", MetadataCleaner())
pipe.add_component("embedder", SentenceTransformersDocumentEmbedder())
pipe.add_component("writer", DocumentWriter(document_store=document_store))